<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>


# Bitcoin Foundations
## Chapter 9 – Privacy, Attacks, and Real-World Risks

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH<br>
The Crypto Engineer | AI-Powered by GPT 5.1.


This lab constructs a tiny synthetic transaction graph and applies a naive multi-input clustering heuristic. It shows how simple rules can already link addresses together, grounding the chapter’s discussion of on-chain privacy and analyst tooling in a concrete toy example.

In [ ]:
from collections import defaultdict  # grouping helper
from typing import List, Dict, Set  # type hints


Tx = Dict[str, List[str]]  # tx model: {"inputs": [...], "outputs": [...]}\n


def multi_input_clusters(txs: List[Tx]) -> List[Set[str]]:
    """Return clusters of addresses based on shared inputs."""
    parent: Dict[str, str] = {}  # union-find parent map

    def find(x: str) -> str:
        """Find representative for address x."""
        parent.setdefault(x, x)
        if parent[x] != x:
            parent[x] = find(parent[x])  # path compression
        return parent[x]

    def union(a: str, b: str) -> None:
        """Merge clusters containing a and b."""
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra  # attach one root to the other

    # Link all inputs within the same transaction
    for tx in txs:
        inputs = tx["inputs"]
        for i in range(1, len(inputs)):
            union(inputs[0], inputs[i])  # connect to first input

    # Build clusters from parent map
    clusters: Dict[str, Set[str]] = defaultdict(set)
    for addr in parent:
        clusters[find(addr)].add(addr)
    return list(clusters.values())


toy_txs: List[Tx] = [
    {"inputs": ["a1", "a2"], "outputs": ["s", "ca"]},
    {"inputs": ["b1"], "outputs": ["s", "cb"]},
    {"inputs": ["a2", "a3"], "outputs": ["x"]},
]  # end of toy transaction list

for idx, cluster in enumerate(multi_input_clusters(toy_txs), start=1):
    print(f"Cluster {idx}: {sorted(cluster)}")  # show grouped addresses


<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>
